# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and visualizing a dataset defined by a Croissant schema using the `mlcroissant` library. All references to fields, record sets, and columns are made by their `@id`s as per best practices.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load both the Croissant metadata and actual records using `mlcroissant`.

This will provide high-level metadata about the dataset and prepare it for further exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'date_published', 'N/A')}")

## 2. Data Overview
List the available record sets, their `@id`s, and their associated fields and field `@id`s.

We will explicitly reference entities by their `@id` throughout the notebook.

In [ ]:
# List all record sets by their @id
print('\nRecord Sets in the dataset:')
for record_set in metadata.record_sets:
    print(f"- @id: {record_set.id} | name: {record_set.name}")
    # List fields for each record set
    print("  Fields:")
    for field in record_set.fields:
        print(f"      - @id: {field.id} | name: {field.name} | data_type: {field.data_type}")


## 3. Data Extraction
Load the contents of each available record set into pandas DataFrames, using only their `@id`s for references.

Below we extract all records for each record set and store them for manipulation and exploration.

In [ ]:
# Get the list of record set @ids
record_set_ids = [r.id for r in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Use the @id for extraction
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set '@id': {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())


## 4. Exploratory Data Analysis (EDA)
As an example, we will:
- Filter records by a numeric field (e.g., "MW [kDa]" for protein molecular weight) using the field's `@id`.
- Normalize the field.
- Optionally group by a categorical field (e.g., protein description or accession) by `@id`.

Update the code to use the correct `@id` values from your data overview above.

In [ ]:
# Choose a record set to analyze (update this to your target record set's @id from above)
target_record_set_id = record_set_ids[0]
# Print columns again for reference
print(f"Columns in {target_record_set_id}: {dataframes[target_record_set_id].columns.tolist()}")

# Pick a numeric field @id for demonstration (example: 'MW [kDa]' or similar, shown as the actual column name)
numeric_field_id = [col for col in dataframes[target_record_set_id].columns if 'mw' in col.lower() or 'kda' in col.lower()]
if numeric_field_id:
    numeric_field_id = numeric_field_id[0]
else:
    # Fallback: pick any numeric-type column
    for col in dataframes[target_record_set_id].columns:
        if pd.api.types.is_numeric_dtype(dataframes[target_record_set_id][col]):
            numeric_field_id = col
            break

print(f"Selected numeric field (by @id/column): {numeric_field_id}")

# Filter for values above an arbitrary threshold
threshold = 10
if numeric_field_id:
    # Ensure that column is numeric
    numeric_col = pd.to_numeric(dataframes[target_record_set_id][numeric_field_id], errors='coerce')
    filtered_df = dataframes[target_record_set_id][numeric_col > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (first 5 rows):")
    display(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (numeric_col[filtered_df.index] - numeric_col.mean()) / numeric_col.std()
    print(f"Normalized values for {numeric_field_id} (first 5 rows):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optionally group by a categorical field (e.g., by "description" or similar field @id)
    group_field_options = [col for col in filtered_df.columns if 'desc' in col.lower() or 'access' in col.lower()]
    if group_field_options:
        group_field_id = group_field_options[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"Mean {numeric_field_id} by {group_field_id} (first 5 rows):")
        display(grouped_df.head())
else:
    print("No numeric field found for demonstration.")

## 5. Visualization
Visualize the distribution of the filtered and normalized numeric field using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and not filtered_df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, bins=30, color='green')
    plt.title(f"Distribution of Normalized {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id} (normalized)")
    plt.ylabel("Frequency")
    plt.show()

## 6. Conclusion

In this notebook, you learned how to load, explore, process, and visualize a dataset described by a Croissant schema using the `mlcroissant` library. The exploration included working with record set `@id`s and field `@id`s, dynamically manipulating the dataset, and preparing the data for further FAIR research. Adjust the exploration steps as necessary for your particular dataset and analytical goals.